# 🧪 RDKit: Zero to Hero
## A Comprehensive Guide to Cheminformatics in Python

---

**Welcome!** This notebook is your complete guide to RDKit — from absolute basics to expert-level cheminformatics techniques. By the end, you'll be able to:
- Work fluently with molecular structures
- Compute chemical properties and descriptors
- Perform substructure searches and filtering
- Generate molecular fingerprints for similarity searches
- Run virtual screening workflows
- Apply machine learning to molecular data

**Prerequisites:** Basic Python knowledge

**Installation:**
```bash
conda install -c conda-forge rdkit
# OR
pip install rdkit
```

---

## 📚 Table of Contents

1. [Chapter 1: Getting Started](#chapter1)
2. [Chapter 2: Molecular Representations](#chapter2)
3. [Chapter 3: Working with Atoms and Bonds](#chapter3)
4. [Chapter 4: Drawing Molecules](#chapter4)
5. [Chapter 5: Molecular Properties & Descriptors](#chapter5)
6. [Chapter 6: SMARTS & Substructure Search](#chapter6)
7. [Chapter 7: Fingerprints & Similarity](#chapter7)
8. [Chapter 8: Chemical Reactions](#chapter8)
9. [Chapter 9: Conformer Generation & 3D](#chapter9)
10. [Chapter 10: Drug-likeness & ADMET](#chapter10)
11. [Chapter 11: Scaffold & Fragment Analysis](#chapter11)
12. [Chapter 12: Virtual Screening Pipeline](#chapter12)
13. [Chapter 13: Machine Learning with RDKit](#chapter13)
14. [Chapter 14: Advanced Topics](#chapter14)

---
<a id='chapter1'></a>
# Chapter 1: Getting Started 🚀

RDKit is an open-source cheminformatics toolkit written in C++ with Python bindings. It's the industry standard for computational chemistry and drug discovery.

In [ ]:
# Verify your RDKit installation
import rdkit
print(f"RDKit version: {rdkit.__version__}")

# Core modules we'll use throughout this notebook
from rdkit import Chem                          # Core molecule handling
from rdkit.Chem import Draw                     # Drawing utilities
from rdkit.Chem import Descriptors              # Molecular descriptors
from rdkit.Chem import AllChem                  # Extended chemistry (3D, reactions)
from rdkit.Chem import rdMolDescriptors         # More descriptors
from rdkit.Chem import DataStructs              # Fingerprint data structures
from rdkit.Chem import rdFMCS                   # Maximum common substructure
from rdkit.Chem.Draw import IPythonConsole      # Jupyter display integration
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("All imports successful! ✅")

### 1.1 Your First Molecule

**SMILES** (Simplified Molecular Input Line Entry System) is the most common text format for molecules. Think of it as a 1D string encoding of a molecule.

In [ ]:
# Create your first molecule — aspirin!
aspirin = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')

# Check if it was created successfully
if aspirin is not None:
    print("Molecule created successfully!")
    print(f"Number of atoms: {aspirin.GetNumAtoms()}")
    print(f"Number of bonds: {aspirin.GetNumBonds()}")
else:
    print("Invalid SMILES!")

# Display the molecule
aspirin

In [ ]:
# Key rule: ALWAYS check if mol is None before proceeding
def safe_mol(smiles):
    """Safely create a molecule with error handling."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    return mol

# Test with valid and invalid SMILES
valid_smiles   = ['CCO', 'c1ccccc1', 'CC(=O)O']          # ethanol, benzene, acetic acid
invalid_smiles = ['XYZ', 'C(C', 'invalid']                # these will fail

print("Valid SMILES:")
for smi in valid_smiles:
    mol = Chem.MolFromSmiles(smi)
    print(f"  {smi:20s} -> atoms={mol.GetNumAtoms()}, bonds={mol.GetNumBonds()}")

print("\nInvalid SMILES:")
for smi in invalid_smiles:
    mol = Chem.MolFromSmiles(smi)
    print(f"  {smi:20s} -> {'VALID' if mol else 'INVALID'}")

---
<a id='chapter2'></a>
# Chapter 2: Molecular Representations 🔤

Molecules can be represented in many different formats. Understanding conversions between them is essential.

In [ ]:
# ============================================================
# SMILES — most common text format
# ============================================================

# Canonical SMILES: standardized, unique representation
mol = Chem.MolFromSmiles('c1ccc(cc1)O')  # phenol (non-canonical input)
canonical = Chem.MolToSmiles(mol)
print(f"Canonical SMILES: {canonical}")

# Isomeric SMILES: includes stereochemistry
chiral_mol = Chem.MolFromSmiles('C[C@@H](O)F')  # (R)-1-fluoroethanol
isomeric = Chem.MolToSmiles(chiral_mol, isomericSmiles=True)
plain    = Chem.MolToSmiles(chiral_mol, isomericSmiles=False)
print(f"Isomeric SMILES:  {isomeric}")
print(f"Plain SMILES:     {plain}")

In [ ]:
# ============================================================
# InChI and InChIKey — IUPAC standard identifiers
# ============================================================
from rdkit.Chem.inchi import MolToInchi, InchiToInchiKey

aspirin = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')

inchi = MolToInchi(aspirin)
inchikey = InchiToInchiKey(inchi)

print(f"InChI:    {inchi}")
print(f"InChIKey: {inchikey}")
# InChIKey is a fixed-length hash — perfect for database lookups!

In [ ]:
# ============================================================
# Mol blocks (SDF/MOL format) — used in files
# ============================================================
mol_block = Chem.MolToMolBlock(aspirin)
print(mol_block[:300], "...")  # Print first 300 chars

# Read back from mol block
mol_back = Chem.MolFromMolBlock(mol_block)
print(f"\nRead back successfully: {mol_back is not None}")
print(f"Same canonical SMILES: {Chem.MolToSmiles(mol_back) == Chem.MolToSmiles(aspirin)}")

In [ ]:
# ============================================================
# Reading & Writing SDF files (molecule databases)
# ============================================================
import os

# Create a sample SDF with multiple molecules
molecules = [
    ('aspirin',    'CC(=O)Oc1ccccc1C(=O)O'),
    ('caffeine',   'Cn1cnc2c1c(=O)n(c(=O)n2C)C'),
    ('ibuprofen',  'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    ('paracetamol','CC(=O)Nc1ccc(O)cc1'),
]

# Write SDF
writer = Chem.SDWriter('/tmp/drugs.sdf')
for name, smi in molecules:
    mol = Chem.MolFromSmiles(smi)
    mol.SetProp('_Name', name)       # Set molecule name
    mol.SetProp('SMILES', smi)       # Custom property
    writer.write(mol)
writer.close()

# Read SDF
print("Reading SDF file:")
suppl = Chem.SDMolSupplier('/tmp/drugs.sdf')
for mol in suppl:
    if mol is not None:
        name = mol.GetProp('_Name')
        print(f"  {name:15s}: {mol.GetNumAtoms()} atoms, {mol.GetNumBonds()} bonds")

In [ ]:
# ============================================================
# SMARTS — the query language of cheminformatics
# ============================================================
# SMARTS is like a regex for molecules — we'll explore this more in Chapter 6

# SMARTS pattern for carboxylic acid
smarts = Chem.MolFromSmarts('[CX3](=O)[OX2H1]')
print(f"Carboxylic acid SMARTS created: {smarts is not None}")

# Test against molecules
test_mols = {
    'acetic acid':  'CC(=O)O',
    'ethanol':      'CCO',
    'aspirin':      'CC(=O)Oc1ccccc1C(=O)O',
}
for name, smi in test_mols.items():
    mol = Chem.MolFromSmiles(smi)
    has_cooh = mol.HasSubstructMatch(smarts)
    print(f"  {name:15s}: has COOH = {has_cooh}")

---
<a id='chapter3'></a>
# Chapter 3: Working with Atoms and Bonds ⚛️

Molecules are graphs: atoms are nodes, bonds are edges. RDKit lets you traverse and query this graph.

In [ ]:
mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')  # aspirin

# ============================================================
# Inspecting Atoms
# ============================================================
print("ATOM INFORMATION:")
print(f"{'Idx':>4} {'Symbol':>8} {'AtomicNum':>10} {'Degree':>7} {'Aromatic':>10} {'Charge':>8} {'NumHs':>7}")
print("-" * 60)

for atom in mol.GetAtoms():
    print(f"{atom.GetIdx():>4} "
          f"{atom.GetSymbol():>8} "
          f"{atom.GetAtomicNum():>10} "
          f"{atom.GetDegree():>7} "
          f"{str(atom.GetIsAromatic()):>10} "
          f"{atom.GetFormalCharge():>8} "
          f"{atom.GetTotalNumHs():>7}")

In [ ]:
# ============================================================
# Inspecting Bonds
# ============================================================
from rdkit.Chem import rdchem

print("BOND INFORMATION:")
print(f"{'Idx':>4} {'BeginAtom':>10} {'EndAtom':>9} {'BondType':>12} {'Aromatic':>10} {'Conjugated':>11}")
print("-" * 65)

for bond in mol.GetBonds():
    bt = bond.GetBondTypeAsDouble()
    bond_type = {1.0: 'SINGLE', 1.5: 'AROMATIC', 2.0: 'DOUBLE', 3.0: 'TRIPLE'}.get(bt, str(bt))
    print(f"{bond.GetIdx():>4} "
          f"{bond.GetBeginAtomIdx():>10} "
          f"{bond.GetEndAtomIdx():>9} "
          f"{bond_type:>12} "
          f"{str(bond.GetIsAromatic()):>10} "
          f"{str(bond.GetIsConjugated()):>11}")

In [ ]:
# ============================================================
# Ring Information
# ============================================================
mol = Chem.MolFromSmiles('c1ccc2ccccc2c1')  # naphthalene

ring_info = mol.GetRingInfo()
print(f"Number of rings: {ring_info.NumRings()}")
print(f"Atom rings: {ring_info.AtomRings()}")
print(f"Bond rings: {ring_info.BondRings()}")

# Check if an atom is in a ring
for atom in mol.GetAtoms():
    print(f"  Atom {atom.GetIdx()} ({atom.GetSymbol()}): in ring = {atom.IsInRing()}, "
          f"in ring of size 6 = {atom.IsInRingSize(6)}")

In [ ]:
# ============================================================
# Modifying Molecules with RWMol (Read-Write Molecule)
# ============================================================
# RWMol allows you to edit molecules atom by atom

# Start from scratch: build ethanol manually
rw_mol = Chem.RWMol()

# Add atoms
c1_idx = rw_mol.AddAtom(Chem.Atom(6))   # Carbon
c2_idx = rw_mol.AddAtom(Chem.Atom(6))   # Carbon
o_idx  = rw_mol.AddAtom(Chem.Atom(8))   # Oxygen

# Add bonds
rw_mol.AddBond(c1_idx, c2_idx, Chem.BondType.SINGLE)
rw_mol.AddBond(c2_idx, o_idx,  Chem.BondType.SINGLE)

# Sanitize (calculate valences, aromaticity, etc.)
Chem.SanitizeMol(rw_mol)

mol = rw_mol.GetMol()
print(f"Built molecule: {Chem.MolToSmiles(mol)}")  # Should be CCO

In [ ]:
# ============================================================
# Stereochemistry
# ============================================================
from rdkit.Chem import rdchem

# Chiral centers
mol = Chem.MolFromSmiles('C[C@@H](F)Cl')  # (R)-configuration
Chem.AssignStereochemistry(mol)

for atom in mol.GetAtoms():
    cip = atom.GetPropsAsDict().get('_CIPCode', None)
    if cip:
        print(f"Atom {atom.GetIdx()} ({atom.GetSymbol()}): CIP = {cip}")

# E/Z double bonds
mol_ez = Chem.MolFromSmiles('C/C=C/C')  # (E)-but-2-ene
Chem.AssignStereochemistry(mol_ez)
for bond in mol_ez.GetBonds():
    if bond.GetStereo() != rdchem.BondStereo.STEREONONE:
        print(f"Bond {bond.GetIdx()}: stereo = {bond.GetStereo()}")

---
<a id='chapter4'></a>
# Chapter 4: Drawing Molecules 🎨

Visualization is crucial for chemistry. RDKit offers powerful 2D drawing capabilities.

In [ ]:
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG, Image

# Draw a single molecule
mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')

# Simple display (works directly in Jupyter)
Draw.MolToImage(mol, size=(400, 300))

In [ ]:
# Draw multiple molecules in a grid
smiles_list = [
    ('Aspirin',     'CC(=O)Oc1ccccc1C(=O)O'),
    ('Caffeine',    'Cn1cnc2c1c(=O)n(c(=O)n2C)C'),
    ('Ibuprofen',   'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    ('Paracetamol', 'CC(=O)Nc1ccc(O)cc1'),
    ('Morphine',    'OC1=CC=C2CC3N(C)CCC34C2=C1OC4'),
    ('Penicillin',  'CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)Cc3ccccc3)C(=O)O)C'),
]

mols   = [Chem.MolFromSmiles(smi) for _, smi in smiles_list]
names  = [name for name, _ in smiles_list]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=3,
    subImgSize=(400, 300),
    legends=names
)
img

In [ ]:
# ============================================================
# Highlighting Atoms and Bonds
# ============================================================
from rdkit.Chem.Draw import rdMolDraw2D

mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')  # aspirin

# Find carboxylic acid substructure
patt = Chem.MolFromSmarts('[CX3](=O)[OX2H1]')
match = mol.GetSubstructMatch(patt)
print(f"Matching atoms: {match}")

# Draw with highlighting
drawer = rdMolDraw2D.MolDraw2DSVG(400, 300)
drawer.drawOptions().addStereoAnnotation = True
rdMolDraw2D.PrepareAndDrawMolecule(
    drawer, mol,
    highlightAtoms=list(match),
    highlightAtomColors={i: (1, 0.5, 0.5) for i in match}  # light red
)
drawer.FinishDrawing()
SVG(drawer.GetDrawingText())

In [ ]:
# ============================================================
# Save molecules to file
# ============================================================
# PNG
img = Draw.MolToImage(mol, size=(600, 400))
img.save('/tmp/aspirin.png')
print("Saved as PNG")

# SVG (vector, scales infinitely)
drawer = rdMolDraw2D.MolDraw2DSVG(600, 400)
drawer.DrawMolecule(mol)
drawer.FinishDrawing()
with open('/tmp/aspirin.svg', 'w') as f:
    f.write(drawer.GetDrawingText())
print("Saved as SVG")

---
<a id='chapter5'></a>
# Chapter 5: Molecular Properties & Descriptors 📊

Molecular descriptors are numerical representations of molecular properties. They're the bridge between chemistry and machine learning.

In [ ]:
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors

mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')  # aspirin

# ============================================================
# Basic Physical Properties
# ============================================================
print("=== Basic Properties ===")
print(f"Molecular Weight (MW):          {Descriptors.MolWt(mol):.2f} Da")
print(f"Exact MW:                       {Descriptors.ExactMolWt(mol):.4f} Da")
print(f"Molecular Formula:              {rdMolDescriptors.CalcMolFormula(mol)}")
print(f"LogP (Lipophilicity):           {Descriptors.MolLogP(mol):.2f}")
print(f"TPSA (Topological Polar SA):    {Descriptors.TPSA(mol):.2f} Å²")
print(f"H-Bond Donors:                  {Descriptors.NumHDonors(mol)}")
print(f"H-Bond Acceptors:               {Descriptors.NumHAcceptors(mol)}")
print(f"Rotatable Bonds:                {Descriptors.NumRotatableBonds(mol)}")
print(f"Rings:                          {Descriptors.RingCount(mol)}")
print(f"Aromatic Rings:                 {rdMolDescriptors.CalcNumAromaticRings(mol)}")
print(f"Heavy Atoms:                    {Descriptors.HeavyAtomCount(mol)}")
print(f"Fraction Csp3:                  {Descriptors.FractionCSP3(mol):.3f}")

In [ ]:
# ============================================================
# All Available Descriptors (~200 built-in)
# ============================================================
all_desc = Descriptors.descList
print(f"Total descriptors available: {len(all_desc)}")
print("\nFirst 20:")
for name, func in all_desc[:20]:
    print(f"  {name}")

In [ ]:
# ============================================================
# Compute ALL Descriptors at once
# ============================================================
from rdkit.ML.Descriptors import MoleculeDescriptors

desc_names = [x[0] for x in Descriptors.descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)

# Calculate for aspirin
desc_values = calculator.CalcDescriptors(mol)
desc_df = pd.DataFrame([desc_values], columns=desc_names)
print(f"Descriptor vector shape: {desc_df.shape}")
desc_df.iloc[:, :10]  # Show first 10

In [ ]:
# ============================================================
# Compute descriptors for a set of molecules
# ============================================================
drug_smiles = {
    'Aspirin':     'CC(=O)Oc1ccccc1C(=O)O',
    'Caffeine':    'Cn1cnc2c1c(=O)n(c(=O)n2C)C',
    'Ibuprofen':   'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
    'Paracetamol': 'CC(=O)Nc1ccc(O)cc1',
    'Atorvastatin':'CC(C)c1c(C(=O)Nc2ccccc2F)c(c3ccc(F)cc3)n(CC[C@@H](O)C[C@@H](O)CC(=O)O)c1c4ccc(F)cc4',
    'Metformin':   'CN(C)C(=N)NC(=N)N',
    'Morphine':    'OC1=CC=C2CC3N(C)CCC34C2=C1OC4',
}

rows = []
for name, smi in drug_smiles.items():
    mol = Chem.MolFromSmiles(smi)
    if mol:
        rows.append({
            'Name':     name,
            'MW':       round(Descriptors.MolWt(mol), 2),
            'LogP':     round(Descriptors.MolLogP(mol), 2),
            'TPSA':     round(Descriptors.TPSA(mol), 2),
            'HBD':      Descriptors.NumHDonors(mol),
            'HBA':      Descriptors.NumHAcceptors(mol),
            'RotBonds': Descriptors.NumRotatableBonds(mol),
            'Rings':    Descriptors.RingCount(mol),
            'ArRings':  rdMolDescriptors.CalcNumAromaticRings(mol),
            'Fsp3':     round(Descriptors.FractionCSP3(mol), 3),
        })

df = pd.DataFrame(rows).set_index('Name')
df

---
<a id='chapter6'></a>
# Chapter 6: SMARTS & Substructure Search 🔍

SMARTS is a query language for chemical patterns — the most powerful tool for structural filtering.

In [ ]:
# ============================================================
# SMARTS Basics
# ============================================================
# Atom notation:
#   [#6]     = any carbon
#   [CX4]    = sp3 carbon (4 connections)
#   [CX3]    = sp2 carbon (3 connections)
#   [c]      = aromatic carbon
#   [NX3]    = trivalent nitrogen
#   [OX2H1]  = hydroxyl oxygen
#   [!#1]    = not hydrogen
#   [$(C=O)] = carbon in carbonyl

# Common functional group SMARTS
functional_groups = {
    'Carboxylic Acid':    '[CX3](=O)[OX2H1]',
    'Ester':              '[CX3](=O)[OX2][#6]',
    'Amide':              '[CX3](=O)[NX3]',
    'Amine (primary)':    '[NX3;H2;!$(NC=O)]',
    'Amine (secondary)':  '[NX3;H1;!$(NC=O)]',
    'Amine (tertiary)':   '[NX3;H0;!$(NC=O)]',
    'Hydroxyl':           '[OX2H]',
    'Ketone':             '[CX3](=O)[#6]',
    'Aldehyde':           '[CX3H1](=O)',
    'Aromatic Ring':      'c1ccccc1',
    'Nitro Group':        '[N+](=O)[O-]',
    'Sulfonamide':        '[SX4](=O)(=O)[NX3]',
    'Halide':             '[F,Cl,Br,I]',
}

print("Compiled SMARTS patterns:")
for name, smarts in functional_groups.items():
    patt = Chem.MolFromSmarts(smarts)
    status = "✅" if patt else "❌"
    print(f"  {status} {name:25s}: {smarts}")

In [ ]:
# ============================================================
# Substructure Matching
# ============================================================
def find_functional_groups(smiles):
    """Identify functional groups in a molecule."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    found = []
    for name, smarts in functional_groups.items():
        patt = Chem.MolFromSmarts(smarts)
        matches = mol.GetSubstructMatches(patt)
        if matches:
            found.append(f"{name} (x{len(matches)})")
    return found

# Test on our drugs
for name, smi in drug_smiles.items():
    groups = find_functional_groups(smi)
    print(f"\n{name}:")
    for g in groups:
        print(f"  - {g}")

In [ ]:
# ============================================================
# GetSubstructMatches vs HasSubstructMatch
# ============================================================
mol = Chem.MolFromSmiles('OCC(O)CO')  # glycerol (3 OH groups)
oh_patt = Chem.MolFromSmarts('[OX2H]')

# HasSubstructMatch: boolean, fast check
has_oh = mol.HasSubstructMatch(oh_patt)
print(f"Has OH: {has_oh}")

# GetSubstructMatch: first match only (tuple of atom indices)
first_match = mol.GetSubstructMatch(oh_patt)
print(f"First OH match at atoms: {first_match}")

# GetSubstructMatches: ALL matches
all_matches = mol.GetSubstructMatches(oh_patt)
print(f"All OH matches: {all_matches}")
print(f"Count of OH groups: {len(all_matches)}")

In [ ]:
# ============================================================
# PAINS (Pan-Assay Interference Compounds) Filtering
# ============================================================
# PAINS are structural alerts for compounds that interfere with assays
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
catalog = FilterCatalog(params)

test_molecules = {
    'Aspirin (clean)':       'CC(=O)Oc1ccccc1C(=O)O',
    'Rhodanine (PAINS)':     'O=C1NC(=S)SC1',
    'Curcumin (tricky)':     'COc1cc(/C=C/C(=O)/C=C/c2ccc(O)c(OC)c2)ccc1O',
    'Caffeine (clean)':      'Cn1cnc2c1c(=O)n(c(=O)n2C)C',
}

print("PAINS Filter Results:")
for name, smi in test_molecules.items():
    mol = Chem.MolFromSmiles(smi)
    entry = catalog.GetFirstMatch(mol)
    if entry:
        print(f"  ❌ {name}: PAINS alert '{entry.GetDescription()}'")
    else:
        print(f"  ✅ {name}: No PAINS alert")

---
<a id='chapter7'></a>
# Chapter 7: Fingerprints & Molecular Similarity 🔢

Fingerprints encode molecular structure as bit vectors, enabling fast similarity searches across large databases.

In [ ]:
from rdkit.Chem import DataStructs

# ============================================================
# Types of Fingerprints
# ============================================================
mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')  # aspirin

# 1. Morgan (ECFP) Fingerprints — most popular in ML
#    radius=2 → ECFP4, radius=3 → ECFP6
morgan_fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
print(f"Morgan FP:    {morgan_fp.GetNumBits()} bits, {morgan_fp.GetNumOnBits()} on")

# 2. MACCS Keys — 166 predefined keys
from rdkit.Chem import MACCSkeys
maccs_fp = MACCSkeys.GenMACCSKeys(mol)
print(f"MACCS FP:     {maccs_fp.GetNumBits()} bits, {maccs_fp.GetNumOnBits()} on")

# 3. RDKit Fingerprint — topological
rdkit_fp = Chem.RDKFingerprint(mol, minPath=1, maxPath=7, fpSize=2048)
print(f"RDKit FP:     {rdkit_fp.GetNumBits()} bits, {rdkit_fp.GetNumOnBits()} on")

# 4. Topological Torsion
tt_fp = rdMolDescriptors.GetTopologicalTorsionFingerprintAsIntVect(mol)
print(f"TopTorsion:   count-based fingerprint")

# 5. Atom Pair
ap_fp = rdMolDescriptors.GetAtomPairFingerprintAsBitVect(mol)
print(f"Atom Pair FP: {ap_fp.GetNumBits()} bits, {ap_fp.GetNumOnBits()} on")

In [ ]:
# ============================================================
# Tanimoto Similarity — the standard metric
# ============================================================
# Tanimoto = |A ∩ B| / |A ∪ B|  (ranges 0 to 1)

def tanimoto_similarity(smi1, smi2, radius=2, nbits=2048):
    """Compute Tanimoto similarity between two SMILES strings."""
    mol1 = Chem.MolFromSmiles(smi1)
    mol2 = Chem.MolFromSmiles(smi2)
    fp1 = AllChem.GetMorganFingerprintAsBitVect(mol1, radius, nbits)
    fp2 = AllChem.GetMorganFingerprintAsBitVect(mol2, radius, nbits)
    return DataStructs.TanimotoSimilarity(fp1, fp2)

pairs = [
    ('Aspirin',   'CC(=O)Oc1ccccc1C(=O)O'),
    ('Salicylic acid (aspirin metabolite)', 'OC(=O)c1ccccc1O'),
    ('Ibuprofen', 'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    ('Caffeine',  'Cn1cnc2c1c(=O)n(c(=O)n2C)C'),
    ('Benzene',   'c1ccccc1'),
]

ref_name, ref_smi = pairs[0]
print(f"Similarities to {ref_name}:")
for name, smi in pairs[1:]:
    sim = tanimoto_similarity(ref_smi, smi)
    bar = '█' * int(sim * 20)
    print(f"  {name:40s}: {sim:.3f} |{bar}")

In [ ]:
# ============================================================
# Similarity Matrix Heatmap
# ============================================================
import seaborn as sns

drug_list = [
    ('Aspirin',     'CC(=O)Oc1ccccc1C(=O)O'),
    ('Salicylate',  'OC(=O)c1ccccc1O'),
    ('Ibuprofen',   'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    ('Naproxen',    'COc1ccc2cc(ccc2c1)[C@@H](C)C(=O)O'),
    ('Caffeine',    'Cn1cnc2c1c(=O)n(c(=O)n2C)C'),
    ('Theophylline','Cn1cnc2c1c(=O)[nH]c(=O)n2C'),
    ('Theobromine', 'Cn1cnc2c1[nH]c(=O)n(c2=O)C'),
    ('Morphine',    'OC1=CC=C2CC3N(C)CCC34C2=C1OC4'),
]

names = [n for n, _ in drug_list]
fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), 2, 2048) for _, s in drug_list]

# Build similarity matrix
n = len(fps)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])

plt.figure(figsize=(10, 8))
sns.heatmap(sim_matrix, xticklabels=names, yticklabels=names,
            annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1)
plt.title('Tanimoto Similarity Matrix (Morgan FP, ECFP4)')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Converting fingerprints to numpy arrays (for ML)
# ============================================================
def fp_to_array(mol, radius=2, nbits=2048):
    """Convert a molecule to a numpy fingerprint array."""
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nbits)
    arr = np.zeros(nbits, dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

# Example
mol = Chem.MolFromSmiles('CCO')
arr = fp_to_array(mol)
print(f"Array shape: {arr.shape}, dtype: {arr.dtype}")
print(f"Number of 1s: {arr.sum()}")
print(f"First 50 bits: {arr[:50]}")

---
<a id='chapter8'></a>
# Chapter 8: Chemical Reactions ⚗️

RDKit can model and apply chemical reactions using reaction SMARTS (SMIRKS).

In [ ]:
from rdkit.Chem import AllChem

# ============================================================
# Reaction SMARTS (SMIRKS)
# Format: [reactants]>>[products]
# []:R means reactant, []:P means product
# ============================================================

# Ester hydrolysis: ester + water → acid + alcohol
ester_hydrolysis = AllChem.ReactionFromSmarts('[C:1](=O)[O:2][C:3]>>[C:1](=O)[OH].[O:2][C:3]')
print(f"Reaction created: {ester_hydrolysis is not None}")

# Apply to aspirin (which contains an ester linkage)
aspirin = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')
products = ester_hydrolysis.RunReactants((aspirin,))

print(f"Number of product sets: {len(products)}")
for i, prod_set in enumerate(products):
    prod_smiles = [Chem.MolToSmiles(p) for p in prod_set]
    print(f"  Product set {i+1}: {' + '.join(prod_smiles)}")

In [ ]:
# ============================================================
# Common Reaction Library
# ============================================================
reactions = {
    'Amide coupling':      '[C:1](=O)[OH].[N:2]>>[C:1](=O)[N:2]',
    'Reductive amination': '[C:1](=O).[N:2]>>[C:1][N:2]',   # simplified
    'Williamson ether':    '[O:1][H].[Cl][C:2]>>[O:1][C:2]',
    'Aldol':               '[C:1](=O)[C:2].[C:3](=O)>>[C:1](=O)[C:2][C@@H:3]([OH])',  # simplified
    'Ester formation':     '[C:1](=O)[OH].[O:2][H]>>[C:1](=O)[O:2]',
    'N-methylation':       '[N:1][H].[CH3:2][Cl]>>[N:1][CH3:2]',
}

# Amide coupling demo: acetic acid + methylamine → N-methylacetamide
rxn = AllChem.ReactionFromSmarts(reactions['Amide coupling'])
acid  = Chem.MolFromSmiles('CC(=O)O')  # acetic acid
amine = Chem.MolFromSmiles('CN')        # methylamine

products = rxn.RunReactants((acid, amine))
unique_products = set()
for prod_set in products:
    for prod in prod_set:
        try:
            Chem.SanitizeMol(prod)
            unique_products.add(Chem.MolToSmiles(prod))
        except:
            pass

print(f"Amide coupling products:")
for p in unique_products:
    print(f"  {p}")

In [ ]:
# ============================================================
# Enumerating a Virtual Library (Combinatorial Chemistry)
# ============================================================
# Build a small amide library from acids × amines

acids = [
    ('Acetic acid',    'CC(=O)O'),
    ('Benzoic acid',   'OC(=O)c1ccccc1'),
    ('Nicotinic acid', 'OC(=O)c1cccnc1'),
]

amines = [
    ('Methylamine',     'CN'),
    ('Aniline',         'Nc1ccccc1'),
    ('Morpholine',      'C1COCCN1'),
]

amide_rxn = AllChem.ReactionFromSmarts('[C:1](=O)[OH].[N:2]>>[C:1](=O)[N:2]')

library = []
for acid_name, acid_smi in acids:
    for amine_name, amine_smi in amines:
        acid_mol  = Chem.MolFromSmiles(acid_smi)
        amine_mol = Chem.MolFromSmiles(amine_smi)
        prods = amide_rxn.RunReactants((acid_mol, amine_mol))
        
        for prod_set in prods:
            for prod in prod_set:
                try:
                    Chem.SanitizeMol(prod)
                    smi = Chem.MolToSmiles(prod)
                    library.append({
                        'Name':   f"{acid_name} + {amine_name}",
                        'SMILES': smi,
                        'MW':     round(Descriptors.MolWt(prod), 2)
                    })
                except:
                    pass

library_df = pd.DataFrame(library).drop_duplicates('SMILES')
print(f"Virtual library size: {len(library_df)} compounds")
library_df

---
<a id='chapter9'></a>
# Chapter 9: Conformer Generation & 3D Structure 🧬

3D structure is essential for docking, pharmacophore modeling, and understanding binding.

In [ ]:
# ============================================================
# Generate 3D Conformers
# ============================================================
mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')  # aspirin

# Step 1: Add hydrogens (important for 3D!)
mol_h = Chem.AddHs(mol)
print(f"Atoms without H: {mol.GetNumAtoms()}")
print(f"Atoms with H:    {mol_h.GetNumAtoms()}")

# Step 2: Embed molecule (generate initial 3D coordinates)
params = AllChem.EmbedParameters()
params.randomSeed = 42
result = AllChem.EmbedMolecule(mol_h, params)
print(f"Embedding result: {result} (0=success, -1=failure)")

# Step 3: Optimize geometry (MMFF94 force field)
AllChem.MMFFOptimizeMolecule(mol_h)
print(f"Optimized. Number of conformers: {mol_h.GetNumConformers()}")

# Get 3D coordinates
conf = mol_h.GetConformer()
print("\n3D coordinates (first 5 atoms):")
for i in range(min(5, mol_h.GetNumAtoms())):
    pos = conf.GetAtomPosition(i)
    sym = mol_h.GetAtomWithIdx(i).GetSymbol()
    print(f"  Atom {i} ({sym}): x={pos.x:.3f}, y={pos.y:.3f}, z={pos.z:.3f}")

In [ ]:
# ============================================================
# Generate Multiple Conformers (conformational sampling)
# ============================================================
mol = Chem.MolFromSmiles('CCCCCC')  # hexane — lots of conformers!
mol = Chem.AddHs(mol)

# Generate 50 conformers
params = AllChem.EmbedParameters()
params.randomSeed = 42
n_confs = AllChem.EmbedMultipleConfs(mol, numConfs=50, params=params)
print(f"Generated {n_confs} conformers")

# Optimize all conformers
results = AllChem.MMFFOptimizeMoleculeConfs(mol)
energies = [r[1] for r in results if r[0] == 0]  # r[0]=0 means converged

print(f"Min energy: {min(energies):.4f} kcal/mol")
print(f"Max energy: {max(energies):.4f} kcal/mol")

# Get lowest energy conformer
min_conf_id = energies.index(min(energies))
print(f"Best conformer ID: {min_conf_id}")

# Align all conformers to the best one
AllChem.AlignMolConformers(mol)

In [ ]:
# ============================================================
# 3D Descriptors
# ============================================================
mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol)

# 3D shape descriptors
print("3D Descriptors:")
print(f"  PMI1 (smallest moment of inertia): {rdMolDescriptors.CalcPMI1(mol):.4f}")
print(f"  PMI2:                              {rdMolDescriptors.CalcPMI2(mol):.4f}")
print(f"  PMI3 (largest moment of inertia):  {rdMolDescriptors.CalcPMI3(mol):.4f}")
print(f"  Radius of Gyration:                {rdMolDescriptors.CalcRadiusOfGyration(mol):.4f} Å")
print(f"  Asphericity:                       {rdMolDescriptors.CalcAsphericity(mol):.4f}")
print(f"  Eccentricity:                      {rdMolDescriptors.CalcEccentricity(mol):.4f}")
print(f"  InertialShapeFactor:               {rdMolDescriptors.CalcInertialShapeFactor(mol):.4f}")

# SASA (Solvent Accessible Surface Area)
from rdkit.Chem import rdFreeSASA
radii = rdFreeSASA.classifyAtoms(mol)
sasa = rdFreeSASA.CalcSASA(mol, radii)
print(f"  SASA:                              {sasa:.2f} Å²")

---
<a id='chapter10'></a>
# Chapter 10: Drug-likeness & ADMET 💊

Drug-likeness rules help filter compounds likely to be orally bioavailable and safe.

In [ ]:
# ============================================================
# Lipinski's Rule of Five (Ro5)
# Predicts oral bioavailability
# ============================================================
def lipinski_ro5(smiles, verbose=True):
    """
    Apply Lipinski's Rule of Five.
    Returns dict with individual rules and overall pass/fail.
    MW <= 500, LogP <= 5, HBD <= 5, HBA <= 10
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd  = Descriptors.NumHDonors(mol)
    hba  = Descriptors.NumHAcceptors(mol)
    
    rules = {
        'MW <= 500':   (mw <= 500,   mw),
        'LogP <= 5':   (logp <= 5,   logp),
        'HBD <= 5':    (hbd <= 5,    hbd),
        'HBA <= 10':   (hba <= 10,   hba),
    }
    
    violations = sum(1 for passed, _ in rules.values() if not passed)
    passes_ro5 = violations <= 1  # Allow one violation
    
    if verbose:
        print(f"Ro5 Analysis:")
        for rule, (passed, val) in rules.items():
            icon = '✅' if passed else '❌'
            print(f"  {icon} {rule}: {val:.2f}")
        print(f"  Violations: {violations}")
        print(f"  Passes Ro5: {passes_ro5}")
    
    return {'violations': violations, 'passes': passes_ro5, 'MW': mw, 'LogP': logp, 'HBD': hbd, 'HBA': hba}

print("=== Aspirin ===")
lipinski_ro5('CC(=O)Oc1ccccc1C(=O)O')
print()
print("=== Paclitaxel (taxol, large drug) ===")
lipinski_ro5('CC(=O)OC1C(=O)[C@@]2(C)CC[C@H]3OC[C@@]45[C@@H]3[C@@H](OC(=O)c3ccccc3)[C@H](OC(=O)[C@@H](NC(=O)c3ccccc3)[C@@H](O)c3ccccc3)C(C)(C)[C@H]4CC[C@@H]25')

In [ ]:
# ============================================================
# Beyond Ro5: More Drug-likeness Rules
# ============================================================
def drug_likeness_full(smiles):
    """Comprehensive drug-likeness assessment."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    mw       = Descriptors.MolWt(mol)
    logp     = Descriptors.MolLogP(mol)
    hbd      = Descriptors.NumHDonors(mol)
    hba      = Descriptors.NumHAcceptors(mol)
    tpsa     = Descriptors.TPSA(mol)
    rotbonds = Descriptors.NumRotatableBonds(mol)
    rings    = Descriptors.RingCount(mol)
    arom_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
    fsp3     = Descriptors.FractionCSP3(mol)
    heavy    = mol.GetNumHeavyAtoms()
    
    return pd.Series({
        # Lipinski Ro5
        'MW':           round(mw, 2),
        'LogP':         round(logp, 2),
        'HBD':          hbd,
        'HBA':          hba,
        'Ro5_pass':     int(sum([mw>500, logp>5, hbd>5, hba>10]) <= 1),
        
        # Veber rules (oral bioavailability)
        'RotBonds':     rotbonds,
        'TPSA':         round(tpsa, 2),
        'Veber_pass':   int(rotbonds <= 10 and tpsa <= 140),
        
        # Lead-like (Oprea)
        'Leadlike_pass': int(200 <= mw <= 350 and -1 <= logp <= 3.5 and rotbonds <= 7),
        
        # Fragment-like (Congreve: Ro3)
        'Fraglike_pass': int(mw <= 300 and logp <= 3 and hbd <= 3 and hba <= 3),
        
        # Quality metrics
        'Fsp3':         round(fsp3, 3),
        'AromaticRings': arom_rings,
        'HeavyAtoms':   heavy,
    })

results = pd.DataFrame({name: drug_likeness_full(smi) for name, smi in drug_smiles.items()}).T
results

In [ ]:
# ============================================================
# Radar (Spider) Plot for Drug-likeness
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

def radar_plot(name, smiles):
    mol = Chem.MolFromSmiles(smiles)
    
    # Normalize each property to 0-1 scale for display
    categories = ['MW/500', 'LogP/5', 'HBD/5', 'HBA/10', 'TPSA/140', 'RotBonds/10']
    values = [
        min(Descriptors.MolWt(mol) / 500, 1.5),
        min((Descriptors.MolLogP(mol) + 2) / 7, 1.5),  # shifted for negative values
        min(Descriptors.NumHDonors(mol) / 5, 1.5),
        min(Descriptors.NumHAcceptors(mol) / 10, 1.5),
        min(Descriptors.TPSA(mol) / 140, 1.5),
        min(Descriptors.NumRotatableBonds(mol) / 10, 1.5)
    ]
    
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    values_plot = values + [values[0]]
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))
    ax.plot(angles, values_plot, 'b-', linewidth=2)
    ax.fill(angles, values_plot, alpha=0.25)
    ax.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='Ro5 limit')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=9)
    ax.set_ylim(0, 1.5)
    ax.set_title(name, size=12, pad=15)
    plt.tight_layout()
    plt.show()

radar_plot('Aspirin', 'CC(=O)Oc1ccccc1C(=O)O')

---
<a id='chapter11'></a>
# Chapter 11: Scaffold & Fragment Analysis 🏗️

Scaffold analysis helps understand the core structures behind bioactive molecules.

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold

# ============================================================
# Murcko Scaffolds
# ============================================================
nsaids = [
    ('Aspirin',    'CC(=O)Oc1ccccc1C(=O)O'),
    ('Ibuprofen',  'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    ('Naproxen',   'COc1ccc2cc(ccc2c1)[C@@H](C)C(=O)O'),
    ('Diclofenac', 'OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl'),
    ('Indomethacin','COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1'),
]

print("Murcko Scaffold Analysis:")
scaffolds = {}
for name, smi in nsaids:
    mol = Chem.MolFromSmiles(smi)
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    scaffold_smi = Chem.MolToSmiles(scaffold)
    
    # Generic scaffold (all atoms → C, all bonds → single)
    generic = MurckoScaffold.MakeScaffoldGeneric(scaffold)
    generic_smi = Chem.MolToSmiles(generic)
    
    scaffolds[name] = scaffold_smi
    print(f"  {name:15s}: {scaffold_smi}")

# Group by scaffold
from collections import defaultdict
scaffold_groups = defaultdict(list)
for name, scaffold in scaffolds.items():
    scaffold_groups[scaffold].append(name)

print("\nScaffold Groups:")
for scaffold, drugs in scaffold_groups.items():
    print(f"  {scaffold}: {drugs}")

In [ ]:
# ============================================================
# Fragment Decomposition
# ============================================================
from rdkit.Chem import BRICS, Fragments

mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')  # aspirin

# BRICS fragmentation
brics_frags = BRICS.BRICSDecompose(mol)
print("BRICS Fragments of Aspirin:")
for frag in sorted(brics_frags):
    print(f"  {frag}")

# Build new molecules from BRICS fragments
mols_for_brics = [Chem.MolFromSmiles(smi) for _, smi in nsaids]
all_frags = set()
for m in mols_for_brics:
    all_frags.update(BRICS.BRICSDecompose(m))

print(f"\nTotal unique BRICS fragments from {len(nsaids)} NSAIDs: {len(all_frags)}")

In [ ]:
# ============================================================
# Maximum Common Substructure (MCS)
# ============================================================
from rdkit.Chem import rdFMCS

mols = [Chem.MolFromSmiles(smi) for _, smi in nsaids[:3]]

# Find MCS
mcs_result = rdFMCS.FindMCS(
    mols,
    timeout=10,
    atomCompare=rdFMCS.AtomCompare.CompareElements,
    bondCompare=rdFMCS.BondCompare.CompareOrder
)

print(f"MCS SMARTS: {mcs_result.smartsString}")
print(f"MCS atoms:  {mcs_result.numAtoms}")
print(f"MCS bonds:  {mcs_result.numBonds}")

# Visualize MCS matches
mcs_mol = Chem.MolFromSmarts(mcs_result.smartsString)
highlight_atoms = [mol.GetSubstructMatch(mcs_mol) for mol in mols]
img = Draw.MolsToGridImage(
    mols,
    highlightAtomLists=highlight_atoms,
    legends=[n for n, _ in nsaids[:3]],
    subImgSize=(350, 250)
)
img

---
<a id='chapter12'></a>
# Chapter 12: Virtual Screening Pipeline 🔬

Virtual screening combines everything above into a workflow to find drug candidates.

In [ ]:
# ============================================================
# Full Virtual Screening Pipeline
# ============================================================

# Sample compound library
compound_library = [
    ('Compound-001', 'CC(=O)Oc1ccccc1C(=O)O'),
    ('Compound-002', 'Cn1cnc2c1c(=O)n(c(=O)n2C)C'),
    ('Compound-003', 'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    ('Compound-004', 'CC(=O)Nc1ccc(O)cc1'),
    ('Compound-005', 'O=C(O)c1ccccc1O'),
    ('Compound-006', 'COc1ccc2cc(ccc2c1)[C@@H](C)C(=O)O'),
    ('Compound-007', 'CCCCCCCCCCCCCCC'),          # lipophilic fail
    ('Compound-008', 'NC(N)=N'),                   # very polar
    ('Compound-009', 'O=C1NC(=S)SC1'),             # PAINS
    ('Compound-010', 'c1ccc(Nc2ccccc2Cl)c(c1)Cl'), # diclofenac-like
    ('Compound-011', 'NCCCCCCCCCCCCCCCC(=O)O'),    # too flexible
    ('Compound-012', 'Cc1ccc(S(=O)(=O)Nc2ccccn2)cc1'),  # sulfonamide
    ('Compound-013', 'invalid_smiles'),             # bad input
    ('Compound-014', 'CC(C)(C)c1ccc(cc1)C(=O)Nc2ccc(cc2)N3CCOCC3'),
    ('Compound-015', 'OC(=O)[C@@H](N)Cc1ccc(O)cc1'),  # tyrosine
]

# Query molecule (active compound we're searching analogs for)
query_smiles = 'CC(=O)Oc1ccccc1C(=O)O'  # aspirin
query_mol = Chem.MolFromSmiles(query_smiles)
query_fp  = AllChem.GetMorganFingerprintAsBitVect(query_mol, 2, 2048)

# PAINS catalog
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog(params)

print("Virtual Screening Pipeline")
print("=" * 70)
print(f"Library size: {len(compound_library)}")
print(f"Query:        {query_smiles}\n")

results = []
for cmpd_id, smi in compound_library:
    record = {'ID': cmpd_id, 'SMILES': smi}
    
    # Step 1: Parse
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        record['Status'] = 'INVALID'
        results.append(record)
        continue
    
    # Step 2: Calculate properties
    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd  = Descriptors.NumHDonors(mol)
    hba  = Descriptors.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rotb = Descriptors.NumRotatableBonds(mol)
    record.update({'MW': round(mw,2), 'LogP': round(logp,2), 
                   'HBD': hbd, 'HBA': hba, 'TPSA': round(tpsa,2), 'RotBonds': rotb})
    
    # Step 3: Lipinski filter
    ro5_violations = sum([mw>500, logp>5, hbd>5, hba>10])
    if ro5_violations > 1:
        record['Status'] = 'FAIL_Ro5'
        results.append(record)
        continue
    
    # Step 4: PAINS filter
    if pains_catalog.GetFirstMatch(mol):
        record['Status'] = 'FAIL_PAINS'
        results.append(record)
        continue
    
    # Step 5: Similarity to query
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048)
    sim = DataStructs.TanimotoSimilarity(query_fp, fp)
    record['Tanimoto'] = round(sim, 3)
    
    record['Status'] = 'PASS'
    results.append(record)

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Summary
status_counts = results_df['Status'].value_counts()
print("\nScreening Summary:")
for status, count in status_counts.items():
    pct = count / len(results_df) * 100
    bar = '█' * int(pct / 3)
    print(f"  {status:15s}: {count:3d} ({pct:4.1f}%) |{bar}")

# Top hits by similarity
passing = results_df[results_df['Status'] == 'PASS'].sort_values('Tanimoto', ascending=False)
print(f"\nTop Hits (Tanimoto > 0.2):")
top = passing[passing.get('Tanimoto', pd.Series()) >= 0.2] if 'Tanimoto' in passing.columns else passing
print(top[['ID', 'MW', 'LogP', 'Tanimoto']].to_string(index=False))

---
<a id='chapter13'></a>
# Chapter 13: Machine Learning with RDKit 🤖

Combining molecular fingerprints with scikit-learn for predictive modeling.

In [ ]:
# ============================================================
# Dataset: BBBP (Blood-Brain Barrier Penetration)
# ============================================================
# We'll create a mini toy dataset here since we can't download BBBP
# In practice, use: https://moleculenet.org/datasets-1

# Simplified toy dataset: 1 = BBB penetrant, 0 = non-penetrant
# Based on rough experimental/literature values
bbbp_data = [
    # BBB+ (penetrant) — more lipophilic, lower MW
    ('Caffeine',      'Cn1cnc2c1c(=O)n(c(=O)n2C)C',           1),
    ('Ethanol',       'CCO',                                     1),
    ('Diazepam',      'C1CN=C(c2ccccc2Cl)c3cc(Cl)ccc3N1',      1),
    ('Morphine',      'OC1=CC=C2CC3N(C)CCC34C2=C1OC4',         1),
    ('Cocaine',       'COC(=O)[C@@H]1[C@@H]2CC[C@H](C1)N2C',  1),
    ('Propranolol',   'CC(C)NCC(O)COc1cccc2ccccc12',           1),
    ('Haloperidol',   'OC1(CCN(CCC(=O)c2ccc(F)cc2)CC1)c1ccc(Cl)cc1', 1),
    ('Imipramine',    'CN(C)CCCN1c2ccccc2CCc2ccccc21',         1),
    
    # BBB- (non-penetrant) — more polar, higher MW
    ('Aspirin',       'CC(=O)Oc1ccccc1C(=O)O',                 0),
    ('Glucose',       'OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O', 0),
    ('Amoxicillin',   'CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H](N)c3ccc(O)cc3)C(=O)O)C', 0),
    ('Metformin',     'CN(C)C(=N)NC(=N)N',                     0),
    ('Atenolol',      'CC(C)NCC(O)COc1ccc(CC(N)=O)cc1',        0),
    ('Captopril',     'CC(CS)C(=O)N1CCCC1C(=O)O',              0),
    ('Lisinopril',    'NCCCC[C@H](NC(=O)[C@@H](CCc1ccccc1)N)CC(=O)N2CCC[C@H]2C(=O)O', 0),
    ('Penicillin',    'CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)Cc3ccccc3)C(=O)O)C', 0),
]

print(f"Dataset: {len(bbbp_data)} molecules")
print(f"BBB+: {sum(1 for _,_,y in bbbp_data if y==1)}")
print(f"BBB-: {sum(1 for _,_,y in bbbp_data if y==0)}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# ============================================================
# Feature Engineering
# ============================================================
def mol_to_features(smiles, use_fingerprint=True, use_descriptors=True):
    """Convert SMILES to feature vector."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    features = []
    
    if use_fingerprint:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
        arr = np.zeros(1024, dtype=np.float32)
        DataStructs.ConvertToNumpyArray(fp, arr)
        features.append(arr)
    
    if use_descriptors:
        desc = np.array([
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.TPSA(mol),
            Descriptors.NumRotatableBonds(mol),
            Descriptors.FractionCSP3(mol),
            rdMolDescriptors.CalcNumAromaticRings(mol),
        ], dtype=np.float32)
        features.append(desc)
    
    return np.concatenate(features)

# Build feature matrix
X = np.array([mol_to_features(smi) for _, smi, _ in bbbp_data])
y = np.array([label for _, _, label in bbbp_data])
names_ml = [name for name, _, _ in bbbp_data]

print(f"Feature matrix shape: {X.shape}")
print(f"Labels: {y}")

In [ ]:
# ============================================================
# Train Random Forest Classifier
# ============================================================
from sklearn.model_selection import StratifiedKFold

rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Cross-validation (leave-one-out for small datasets)
loo = LeaveOneOut()
cv_scores = cross_val_score(rf, X, y, cv=loo, scoring='accuracy')
print(f"LOO-CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train final model
rf.fit(X, y)
train_preds = rf.predict(X)
print(f"Training Accuracy: {accuracy_score(y, train_preds):.3f}")
print()
print(classification_report(y, train_preds, target_names=['BBB-', 'BBB+']))

In [ ]:
# ============================================================
# Predict new compounds
# ============================================================
new_compounds = [
    ('Ibuprofen',    'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    ('Paracetamol',  'CC(=O)Nc1ccc(O)cc1'),
    ('Nicotine',     'CN1CCC[C@H]1c1cccnc1'),
    ('Dopamine',     'NCCc1ccc(O)c(O)c1'),
]

print("BBB Penetration Predictions:")
for name, smi in new_compounds:
    feats = mol_to_features(smi).reshape(1, -1)
    pred = rf.predict(feats)[0]
    prob = rf.predict_proba(feats)[0]
    label = 'BBB+' if pred == 1 else 'BBB-'
    confidence = max(prob)
    print(f"  {name:15s}: {label} (confidence: {confidence:.2%})")

In [ ]:
# ============================================================
# Chemical Space Visualization with PCA
# ============================================================
from sklearn.decomposition import PCA

# Fingerprint-only for PCA
X_fp = np.array([
    mol_to_features(smi, use_fingerprint=True, use_descriptors=False) 
    for _, smi, _ in bbbp_data
])

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_fp)

plt.figure(figsize=(10, 7))
colors = ['red' if yi == 0 else 'blue' for yi in y]
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.8, s=100, edgecolors='black')

for i, name in enumerate(names_ml):
    plt.annotate(name, (X_pca[i, 0], X_pca[i, 1]), 
                fontsize=8, xytext=(5, 5), textcoords='offset points')

from matplotlib.patches import Patch
plt.legend(handles=[Patch(color='blue', label='BBB+'), Patch(color='red', label='BBB-')])
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('Chemical Space (PCA of Morgan Fingerprints)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
<a id='chapter14'></a>
# Chapter 14: Advanced Topics 🚀

Expert-level techniques: pharmacophores, diversity selection, matched molecular pairs.

In [ ]:
# ============================================================
# 14.1 Pharmacophore Features
# ============================================================
from rdkit.Chem import MolChemicalFeatures
from rdkit import RDConfig
import os

# Load feature factory (defines what counts as HBD, HBA, Aromatic, etc.)
fdefName = os.path.join(RDConfig.RDDataDir, 'BaseFeatures.fdef')
factory = MolChemicalFeatures.BuildFeatureFactory(fdefName)

mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')  # aspirin
feats = factory.GetFeaturesForMol(mol)

print(f"Pharmacophoric features of Aspirin:")
for feat in feats:
    print(f"  {feat.GetFamily():20s} at atoms {feat.GetAtomIds()}")

In [ ]:
# ============================================================
# 14.2 Diversity Selection with MaxMin Algorithm
# ============================================================
from rdkit.SimDivFilters import rdSimDivPickers

# Create a pool of molecules
pool_smiles = [
    'CC(=O)Oc1ccccc1C(=O)O',      # aspirin
    'Cn1cnc2c1c(=O)n(c(=O)n2C)C', # caffeine
    'CC(C)Cc1ccc(cc1)C(C)C(=O)O', # ibuprofen
    'CC(=O)Nc1ccc(O)cc1',          # paracetamol
    'COc1ccc2cc(ccc2c1)[C@@H](C)C(=O)O', # naproxen
    'OC(=O)c1ccccc1O',             # salicylic acid
    'c1ccc2ccccc2c1',              # naphthalene
    'CCO',                          # ethanol
    'CCCCC',                        # pentane
    'c1ccc(cc1)N',                  # aniline
]

pool_mols = [Chem.MolFromSmiles(s) for s in pool_smiles]
pool_fps  = [AllChem.GetMorganFingerprintAsBitVect(m, 2, 1024) for m in pool_mols]

# Select 4 diverse molecules using MaxMin picker
picker = rdSimDivPickers.MaxMinPicker()

def distance_fn(i, j):
    return 1.0 - DataStructs.TanimotoSimilarity(pool_fps[i], pool_fps[j])

selected_indices = list(picker.LazyPick(distance_fn, len(pool_fps), 4, seed=42))
print(f"Selected {len(selected_indices)} diverse molecules: indices {selected_indices}")

selected_mols = [pool_mols[i] for i in selected_indices]
Draw.MolsToGridImage(selected_mols, molsPerRow=4, subImgSize=(300, 200),
                     legends=[f"Compound {i}" for i in selected_indices])

In [ ]:
# ============================================================
# 14.3 Matched Molecular Pairs (MMP)
# ============================================================
# MMPs reveal SAR: which structural changes affect which properties

# Simplified MMP demonstration
# In practice, use: rdkit.Chem.MolStandardize or mmpdb library

# Compare aspirin vs salicylic acid (remove acetyl)
mol_a = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')   # aspirin
mol_b = Chem.MolFromSmiles('OC(=O)c1ccccc1O')           # salicylic acid

props = ['MW', 'LogP', 'TPSA', 'HBD', 'HBA']
funcs = [
    Descriptors.MolWt, Descriptors.MolLogP, 
    Descriptors.TPSA, Descriptors.NumHDonors, Descriptors.NumHAcceptors
]

print("Matched Molecular Pair Analysis:")
print(f"  Pair: Aspirin → Salicylic Acid (remove -COCH3)")
print(f"  {'Property':10s} {'Aspirin':>12s} {'Salicylate':>12s} {'Delta':>10s}")
print("  " + "-" * 48)
for prop, fn in zip(props, funcs):
    va = fn(mol_a)
    vb = fn(mol_b)
    delta = vb - va
    print(f"  {prop:10s} {va:12.2f} {vb:12.2f} {delta:+10.2f}")

In [ ]:
# ============================================================
# 14.4 Working with Large Databases (Performance Tips)
# ============================================================
import time

# Generate a mini "large" library
base_smiles = ['CCO', 'CCN', 'CCC', 'c1ccccc1', 'CC=O', 'CCOO']
big_library = base_smiles * 1000  # 6000 molecules

print(f"Processing {len(big_library)} molecules...")

# Slow approach: one by one
t0 = time.time()
results_slow = []
for smi in big_library:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        results_slow.append(Descriptors.MolWt(mol))
t_slow = time.time() - t0

# Fast approach: pre-compile patterns, batch operations
t0 = time.time()
# Pre-parse all mols
mols = [Chem.MolFromSmiles(s) for s in big_library]
valid_mols = [m for m in mols if m is not None]
# Vectorized descriptor calculation
results_fast = [Descriptors.MolWt(m) for m in valid_mols]
t_fast = time.time() - t0

print(f"Both approaches computed {len(results_slow)} values")
print(f"Sequential: {t_slow:.3f}s")
print(f"Batch:      {t_fast:.3f}s")
print(f"\nTips for large databases:")
print("  1. Pre-compile SMARTS patterns outside loops")
print("  2. Use SDMolSupplier for file streaming (avoids loading all into RAM)")
print("  3. Use multiprocessing for embarrassingly parallel tasks")
print("  4. Cache expensive calculations")
print("  5. Use Chem.SDMolSupplier with removeHs=True when H not needed")

In [ ]:
# ============================================================
# 14.5 Multiprocessing for Large-Scale Processing
# ============================================================
from multiprocessing import Pool
import os

def compute_properties(smiles):
    """Worker function — must be picklable (top-level function)."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        'SMILES': smiles,
        'MW':     Descriptors.MolWt(mol),
        'LogP':   Descriptors.MolLogP(mol),
        'TPSA':   Descriptors.TPSA(mol),
    }

# Use multiprocessing
smiles_chunk = big_library[:500]
t0 = time.time()
with Pool(processes=min(4, os.cpu_count())) as pool:
    results_mp = pool.map(compute_properties, smiles_chunk)
t_mp = time.time() - t0

valid_results = [r for r in results_mp if r]
print(f"Multiprocessing: processed {len(valid_results)} mols in {t_mp:.3f}s")
print(f"Results sample:")
pd.DataFrame(valid_results[:5])

In [ ]:
# ============================================================
# 14.6 Standardization and Salt Stripping
# ============================================================
from rdkit.Chem.MolStandardize import rdMolStandardize

def standardize_molecule(smiles):
    """Standardize a molecule: remove salts, neutralize, canonicalize."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Remove salt fragments (keep the largest fragment)
    remover = rdMolStandardize.LargestFragmentChooser()
    mol = remover.choose(mol)
    
    # Uncharge (neutralize)
    uncharger = rdMolStandardize.Uncharger()
    mol = uncharger.uncharge(mol)
    
    # Sanitize
    Chem.SanitizeMol(mol)
    
    return Chem.MolToSmiles(mol)

test_cases = [
    ('Aspirin sodium salt', 'CC(=O)Oc1ccccc1C(=O)[O-].[Na+]'),
    ('HCl salt of amine',   'CCN(CC)CC.Cl'),
    ('Charged molecule',    'CC[N+](C)(C)CC'),
    ('Clean molecule',      'CCO'),
]

print("Standardization Results:")
for name, smi in test_cases:
    std = standardize_molecule(smi)
    print(f"  {name:30s}")
    print(f"    Input:  {smi}")
    print(f"    Output: {std}")

In [ ]:
# ============================================================
# 14.7 QED — Quantitative Estimate of Drug-likeness
# ============================================================
from rdkit.Chem import QED

print("QED Drug-likeness Scores (0=drug-unlike, 1=drug-like):")
for name, smi in drug_smiles.items():
    mol = Chem.MolFromSmiles(smi)
    qed_score = QED.qed(mol)
    bar = '█' * int(qed_score * 20)
    print(f"  {name:15s}: {qed_score:.3f} |{bar}")

# Detailed QED breakdown
print("\nDetailed QED for Aspirin:")
aspirin = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')
qed_props = QED.properties(aspirin)
print(f"  {qed_props}")
print(f"  QED score: {QED.qed(aspirin):.4f}")

In [ ]:
# ============================================================
# 14.8 Generating Molecules with RECAP
# ============================================================
from rdkit.Chem import Recap

mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')  # aspirin

# RECAP decomposition
recap_tree = Recap.RecapDecompose(mol)

print("RECAP Decomposition of Aspirin:")
leaves = recap_tree.GetLeaves()
for smi, node in leaves.items():
    print(f"  {smi}")

print(f"\nNumber of leaf fragments: {len(leaves)}")

---

# 🎓 Summary & Key Takeaways

Congratulations! You've gone from zero to hero with RDKit. Here's what you've mastered:

| Topic | Key Functions |
|-------|---------------|
| Molecule creation | `Chem.MolFromSmiles()`, `Chem.MolFromSmarts()` |
| Format conversion | `Chem.MolToSmiles()`, `MolToInchi()`, `MolToMolBlock()` |
| Atom/bond traversal | `.GetAtoms()`, `.GetBonds()`, atom properties |
| Drawing | `Draw.MolToImage()`, `rdMolDraw2D`, highlighting |
| Descriptors | `Descriptors.*`, `rdMolDescriptors.*`, QED |
| Substructure | `HasSubstructMatch()`, `GetSubstructMatches()`, SMARTS |
| Fingerprints | `GetMorganFingerprintAsBitVect()`, MACCS, RDKit FP |
| Similarity | `DataStructs.TanimotoSimilarity()` |
| Reactions | `AllChem.ReactionFromSmarts()`, `RunReactants()` |
| 3D Conformers | `EmbedMolecule()`, `MMFFOptimizeMolecule()` |
| Drug-likeness | Lipinski Ro5, Veber, PAINS, QED |
| Scaffolds | `MurckoScaffold`, BRICS, RECAP, MCS |
| ML | Fingerprints → numpy → scikit-learn |
| Advanced | Pharmacophores, diversity selection, standardization |

## Next Steps
- 📖 [RDKit Documentation](https://www.rdkit.org/docs/)
- 📦 [MoleculeNet datasets](https://moleculenet.org/) for ML benchmarks
- 🔬 [ZINC database](https://zinc.docking.org/) for large-scale screening
- 🐍 Explore: `deepchem`, `mmpdb`, `prolif`, `datamol`, `espsim`
- 📊 [The RDKit Blog](https://greglandrum.github.io/rdkit-blog/) by Greg Landrum (creator)

---
*Happy molecule hunting! 🧬*